# The Anatomy of AI · Lecture 30b · Already Clean? II: The Verdict
### The notebook

Lecture 30b weighed Goodfire's essay *Neural Networks Are Already Clean* against everything this course measured. The
essay's picture: networks learn mostly *clean* neurons, one job each, and the messy-looking ones are a few readable
mechanisms, not a knot. Here you rebuild three of the lecture's decisive tests yourself:

- **Part A · No room.** What does a network do when it has fewer neurons than things to say? The toy from Lecture 15,
  rebuilt in a few lines.
- **Part B · Language neurons.** A real language model, Qwen3-1.7B: the two halves of its SwiGLU neurons, the essay's
  "noisy middle", a neuron that says *don't say "."*, and what weak activations are worth.
- **Part C · The ledger.** The lecture's 23 results from across the course, tallied.

Every result the lecture states is checked with `check(...)`: ✓, or the notebook stops. The lecture measured Qwen3 on
40 million tokens; here we use a few hundred FineWeb-Edu documents, so where a number depends on the sample the check
allows for it. Five **Your turn** exercises; solutions at the end.

It runs on free Colab in about 15 minutes on a **T4 GPU**. It downloads Qwen3-1.7B (3.4 GB) and about 1 MB of text. No keys.

In [ ]:
# Pinned installs. Free Colab already has torch and torchvision; a machine without them gets the pinned pair.
import importlib.util, subprocess, sys
def pip(*args):
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "--disable-pip-version-check", *args])
if importlib.util.find_spec("torch") is None:
    pip("torch==2.5.1", "torchvision==0.20.1")
pip("numpy==2.0.2", "matplotlib==3.10.0", "scipy==1.14.1", "pillow==11.1.0", "requests==2.32.3", "transformers==4.51.3")
pip("--no-deps", "torch-lucent==0.1.8")       # only its InceptionV1 port is used

In [ ]:
import io, json, math, os, time, warnings
os.environ["HF_HUB_DISABLE_PROGRESS_BARS"] = "1"       # the page shows results, not download bars
os.environ["TRANSFORMERS_VERBOSITY"] = "error"
warnings.filterwarnings("ignore", category=FutureWarning)
import numpy as np
import matplotlib.pyplot as plt
import torch

DEV = "cuda" if torch.cuda.is_available() else "cpu"
print("running on", DEV)

# The course's look: black ink on paper, no colour except real photos.
PAPER, INK, INK2 = "#F1EBDD", "#16130F", "#7A7062"
plt.rcParams.update({
    "figure.facecolor": PAPER, "axes.facecolor": PAPER, "savefig.facecolor": PAPER,
    "axes.edgecolor": INK, "axes.labelcolor": INK, "xtick.color": INK, "ytick.color": INK, "text.color": INK,
    "font.family": "serif", "font.size": 11, "axes.spines.top": False, "axes.spines.right": False,
    "hatch.color": INK, "hatch.linewidth": 0.6, "figure.dpi": 110, "legend.frameon": False,
})


def minus(text):
    """Print negative numbers with a real minus sign, as the lecture does."""
    return str(text).replace("-", "\u2212")


def check(name, value, expected, tol=1e-9):
    """Compare a result with the lecture's own number. Prints ✓, or stops the notebook with ✗."""
    if isinstance(expected, (int, float)) and not isinstance(expected, bool):
        ok = abs(float(value) - expected) <= tol
        shown = f"{float(value):.6g}"
    else:
        ok = value == expected
        shown = repr(value)
    print(f"{'✓' if ok else '✗'} {name}: {minus(shown)}   (lecture: {minus(expected)}{'' if tol < 1e-6 or not isinstance(expected, (int, float)) else f' ± {tol:g}'})")
    if not ok:
        raise AssertionError(f"{name}: got {shown}, the lecture says {expected}")

## Part A · No room

## Plate II · What would superposition do?

Squeeze a network until it has fewer neurons than things it needs to represent. One possibility: it **packs** several
features into the same neurons, at angles to each other (**superposition**). The other: it keeps only the most important
features and lets the rest go, or makes **coarse** neurons that each cover a broad family. The essay bets on coarse; the
lecture squeezed InceptionV1's layer 3b from 480 neurons to 10 and found coarse neurons, none of them a mixture.

But packing is real, in the right conditions. Lecture 15's toy (Elhage et al., 2022) shows exactly when. Five features
x₁…x₅, each zero most of the time (with probability S, the **sparsity**), squeezed into two neurons: h = W x, then
x′ = ReLU(Wᵀh + b). The loss weighs feature i by its importance 0.9ⁱ. Here is the whole model, trained 30 times at
once (three sparsities × ten seeds), keeping the best seed of each.

In [ ]:
def train_toy(S_list=(0.0, 0.8, 0.9), seeds=10, n=5, m=2, steps=20000, batch=2048, lr=1e-3):
    """W: (R, m, n), b: (R, n) for R = len(S_list) x seeds independent models trained as one tensor."""
    R = len(S_list) * seeds
    g = torch.Generator(device="cpu").manual_seed(0)
    W = (torch.randn(R, m, n, generator=g) * math.sqrt(2 / (m + n))).to(DEV).requires_grad_(True)
    b = torch.zeros(R, n, device=DEV, requires_grad=True)
    I = (0.9 ** torch.arange(n, device=DEV)).expand(R, n)
    P = torch.tensor([1 - S for S in S_list for _ in range(seeds)], device=DEV)[:, None]  # chance a feature is on
    opt = torch.optim.Adam([W, b], lr=lr)
    sched = torch.optim.lr_scheduler.LambdaLR(opt, lambda s: min(1.0, (s + 1) / (0.05 * steps))
                                              * (0.05 + 0.95 * 0.5 * (1 + math.cos(math.pi * s / steps))))
    def sample(B):
        x = torch.rand(R, B, n, device=DEV)
        return x * (torch.rand(R, B, n, device=DEV) < P[:, :, None])
    for step in range(steps):
        x = sample(batch)
        h = torch.einsum("rmn,rbn->rbm", W, x)                              # squeeze: 5 numbers -> 2
        out = torch.relu(torch.einsum("rmn,rbm->rbn", W, h) + b[:, None])     # and back out
        loss = (I[:, None] * (x - out) ** 2).sum(-1).mean(-1)                  # one loss per model
        opt.zero_grad(); loss.sum().backward(); opt.step(); sched.step()
    with torch.no_grad():                                                       # judge every model on fresh samples
        x = sample(32768)
        out = torch.relu(torch.einsum("rmn,rbm->rbn", W, torch.einsum("rmn,rbn->rbm", W, x)) + b[:, None])
        final = (I[:, None] * (x - out) ** 2).sum(-1).mean(-1)
    best = {}
    for k, S in enumerate(S_list):
        r = k * seeds + int(final[k * seeds:(k + 1) * seeds].argmin())
        best[S] = (W[r].detach().cpu().numpy(), b[r].detach().cpu().numpy())
    return best


t0 = time.time()
toy = train_toy()
print(f"trained 30 toy models in {time.time() - t0:.0f} s")
kept = {S: int((np.linalg.norm(Wb[0], axis=0) > 0.5).sum()) for S, Wb in toy.items()}
for S, k in kept.items():
    print(f"sparsity {S}: {k} of 5 features kept (|W_i| > 0.5)")
check("features kept with no sparsity", kept[0.0], 2)
check("features kept at 80% sparsity (two opposite pairs)", kept[0.8], 4)
check("features kept at 90% sparsity (a pentagon)", kept[0.9], 5)
W9 = toy[0.9][0]
ang = np.sort(np.degrees(np.arctan2(W9[1], W9[0])) % 360)
gaps = np.diff(np.concatenate([ang, ang[:1] + 360]))
check("the pentagon's mean gap between neighbours", gaps.mean(), 72.0, tol=0.5)
check("its most uneven gap", np.abs(gaps - 72).max(), 0, tol=6)

In [ ]:
fig, ax = plt.subplots(1, 3, figsize=(10, 3.4))
for a, (S, (W_, b_)) in zip(ax, toy.items()):
    for i in range(5):
        a.annotate("", xy=W_[:, i], xytext=(0, 0), arrowprops=dict(arrowstyle="-|>", color=INK, lw=1.6 if np.linalg.norm(W_[:, i]) > 0.5 else 0.6))
        a.text(*(W_[:, i] * 1.15), f"x{i + 1}", ha="center", va="center", fontsize=9)
    a.set_xlim(-1.5, 1.5); a.set_ylim(-1.5, 1.5); a.set_aspect("equal"); a.axhline(0, color=INK2, lw=0.5); a.axvline(0, color=INK2, lw=0.5)
    a.set_title(f"sparsity {S:.0%}: {kept[S]} kept", fontsize=10); a.set_xticks([]); a.set_yticks([])
plt.suptitle("Five features in two neurons: each arrow is one feature's direction", fontsize=10); plt.tight_layout(); plt.show()

Dense features: the two most important get a neuron each, the rest are dropped. Sparse features: all five are packed in
at about 72° from each other, a pentagon, and each neuron now answers to four of the five. That *is* superposition, and
it happens only when features are rare and the network truly has no room.

The lecture's squeeze asked the same question of a real network trained on real photos. InceptionV1's layer 3b,
squeezed from 480 neurons to 10 and retrained from scratch, still reached **64.8%** top-1 at 224 px and **57.7%** at 128
px (the essay's own squeeze: 58.4%). Its ten neurons were coarse (dark circles, red, slanted edges …), none of them a
mixture, and the curve detectors were gone. Real networks squeezed hard went coarse, not packed.

### Your turn 1 · When does the fifth feature join?

Lecture 15 found the fifth feature joins between S = 0.80 and 0.84. Train the toy at a few sparsities between 0.8 and 0.9
(`train_toy(S_list=(...), seeds=10)`) and find the first one that keeps all five.

*Hint: count columns of W with length over 0.5, as above.*

In [ ]:
# TODO: train_toy at sparsities 0.80, 0.82, 0.84, 0.86, 0.88 and count the features kept

## Part B · Language neurons

## Plate VII · Two halves (SwiGLU)

A modern language model's MLP neuron has two halves. For the token's vector x:

- the **gate** g = x · w_gate, passed through SiLU(g) = g · sigmoid(g), which is near 0 for negative g and near g for
  positive g;
- the **value** (`up`) u = x · w_up;
- the neuron's output is **a = SiLU(g) × u**.

So a can be positive two ways: gate open and value positive (+×+), or gate *closed* and value negative (−×−: SiLU of a
negative number is a small negative number). The essay calls the second the "double-negative" regime. Below we load
Qwen3-1.7B and build one neuron's output by hand.

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer
from transformers.utils import logging as hf_logging
hf_logging.disable_progress_bar()                           # no "loading shards" bar on the page
t0 = time.time()
tok = AutoTokenizer.from_pretrained("Qwen/Qwen3-1.7B")
lm = AutoModelForCausalLM.from_pretrained("Qwen/Qwen3-1.7B", torch_dtype=torch.bfloat16).to(DEV).float().eval()
for p in lm.parameters():
    p.requires_grad_(False)
LAYERS = lm.model.layers
print(f"Qwen3-1.7B loaded (a 3.4 GB download) in {time.time() - t0:.0f} s: {len(LAYERS)} layers x {LAYERS[0].mlp.gate_proj.out_features} neurons")

### Some text

The lecture read 40 million tokens. We read a few hundred documents of FineWeb-Edu (educational web pages), fetched by
row from Hugging Face's dataset viewer, and cut them into sequences of 512 tokens.

In [ ]:
import requests
URL = ("https://datasets-server.huggingface.co/rows?dataset=HuggingFaceFW/fineweb-edu&config=sample-10BT"
       "&split=train&offset={}&length=100")
docs = []
for off in (0, 100, 200, 300):
    for k in range(6):
        try:
            docs += [r["row"]["text"] for r in requests.get(URL.format(off), timeout=60).json()["rows"]]
            break
        except Exception:
            time.sleep(2 * (k + 1))
ids = []
for d in docs:
    ids += tok(d)["input_ids"] + [tok.eos_token_id]
SEQ = 512
seqs = torch.tensor(ids[: len(ids) // SEQ * SEQ]).view(-1, SEQ)
print(f"{len(docs)} documents, {seqs.numel():,} tokens, {len(seqs)} sequences of {SEQ}")
SCAN, CALIB, HELD = seqs[:96], seqs[96:112], seqs[112:128]     # three separate sets

In [ ]:
# One neuron by hand: layer 23, neuron 1508 (the lecture's "resources" neuron), on one sequence.
L, n = 23, 1508
mlp = LAYERS[L].mlp
grab = {}
h = mlp.register_forward_pre_hook(lambda m, inp: grab.update(x=inp[0]))
h2 = mlp.down_proj.register_forward_pre_hook(lambda m, inp: grab.update(a=inp[0]))
with torch.no_grad():
    lm(SCAN[:1].to(DEV))
h.remove(); h2.remove()
x = grab["x"][0]                                             # 512 tokens x 2048
g = x @ mlp.gate_proj.weight[n]                              # the gate half
u = x @ mlp.up_proj.weight[n]                                # the value half
a = g * torch.sigmoid(g) * u                                 # SiLU(g) x u
check("largest difference, by hand vs the model", float((a - grab["a"][0, :, n]).abs().max()), 0, tol=1e-3)
print("tokens where the gate is closed:", f"{float((g < 0).float().mean()):.0%}")

## Plate VIII · How rare is strong? The noisy middle

Now every neuron at once: for all 28 × 6,144 = 172,032 neurons and every token of 96 sequences (49,152 tokens), we sort
each value into its **sign regime** (which sign the gate has, which sign the value has) and count. The lecture, over 40M
tokens: −×− 40.2%, −×+ 40.1%, +×− 9.9%, +×+ 9.9%. The gate is closed on about 80% of tokens.

In [ ]:
counts = torch.zeros(len(LAYERS), 6144, 4, device=DEV)        # per neuron: -x-, -x+, +x-, +x+
cur = {}
hooks = []
for li, layer in enumerate(LAYERS):
    hooks.append(layer.mlp.gate_proj.register_forward_hook(lambda m, i, o, li=li: cur.__setitem__(("g", li), o)))
    def count(m, i, o, li=li):
        r = 2 * (cur[("g", li)] > 0).long() + (o > 0).long()  # 0 -x-, 1 -x+, 2 +x-, 3 +x+
        r = r[:, 1:].reshape(-1, 6144)                         # skip position 0, as the lecture does
        for k in range(4):
            counts[li, :, k] += (r == k).sum(0)
    hooks.append(layer.mlp.up_proj.register_forward_hook(count))
t0 = time.time()
with torch.no_grad():
    for s in range(0, len(SCAN), 4):
        lm(SCAN[s:s + 4].to(DEV))
for h in hooks:
    h.remove()
print(f"counted in {time.time() - t0:.0f} s")
tot = counts.sum((0, 1))
shares = (tot / tot.sum()).cpu().numpy()
for name, v, lec in zip(["-x-", "-x+", "+x-", "+x+"], shares, [0.4015, 0.4010, 0.0987, 0.0988]):
    check(f"share of all values in {name}", v, lec, tol=0.03)
check("gate closed (the two minus-gate regimes)", shares[0] + shares[1], 0.8026, tol=0.03)

The gate is closed on most tokens, and then a = SiLU(g) × u is small with the sign of u. So most of a neuron's
mildly-positive values are −×−: background, not the neuron's job. The essay's claim is that the neuron's *job* lives in
the tail, where the gate is open. The lecture's showcase, layer 23 neuron 1508: over 40M tokens, its band from the 75th
to the 99th percentile is **95.6%** −×−, while every one of its strongest values (≥ the 99.99th percentile) is +×+. We use
the lecture's own percentile thresholds and our tokens.

In [ ]:
T = {"p75": 0.0731, "p99": 2.347, "p99.9": 28.5259, "p99.99": 95.3764, "p25": -0.0166, "p1": -1.5949, "p0.1": -12.3966, "p0.01": -43.8724}       # neuron 1508's percentiles over the lecture's 40M tokens
vals, regs, where = [], [], []
def keep(m, inp):
    vals.append(inp[0][:, 1:, 1508].float().cpu())
mlp = LAYERS[23].mlp
hk = [mlp.down_proj.register_forward_pre_hook(keep),
      mlp.gate_proj.register_forward_hook(lambda m, i, o: regs.append((o[:, 1:, 1508] > 0).cpu()))]
with torch.no_grad():
    for s in range(0, len(SCAN), 4):
        lm(SCAN[s:s + 4].to(DEV))
for h in hk:
    h.remove()
A, G = torch.cat(vals).flatten(), torch.cat(regs).flatten()
mid = (A > T["p75"]) & (A < T["p99"])
tail = A >= T["p99.99"]
print(f"{int(mid.sum()):,} tokens in the p75-p99 band, {int(tail.sum())} at or above p99.99")
check("p75-p99 band: share with the gate closed (-x-)", float((~G[mid]).float().mean()), 0.9557, tol=0.03)
check("at or above p99.99: share with the gate open (+x+)", float(G[tail].float().mean()), 1.0, tol=1e-9)
# its strongest tokens, in context
order = torch.argsort(A, descending=True)[:6]
flat = SCAN[:, 1:].flatten()
for i in order.tolist():
    ctx = tok.decode(flat[max(0, i - 10): i]); here = tok.decode(flat[i:i + 1])
    print(f"{float(A[i]):7.1f}  …{ctx!r} ⟦{here}⟧")

Its strongest tokens should be about natural resources (the lecture's: "huge minerals deposits", "abundant natural
resource"); its middle band is unrelated words with the gate shut.

### Your turn 2 · Is the neuron typical?

The lecture found the "enemy" pattern (at least 90% −×− in p75–p99, at most 1% −×− at the very top) in 44.8% of all
neurons, so 1508 is not special. Using `counts`, compute for each layer the share of neurons whose gate is open on fewer
than 10% of tokens. Which layers have the most nearly-closed gates?

*Hint: the gate-open share of a neuron is `(counts[..., 2] + counts[..., 3]) / counts.sum(-1)`.*

In [ ]:
# TODO: per layer, the share of neurons whose gate opens on fewer than 10% of tokens

## Plate XI · Don't say "."

The essay's favourite small mechanism: a neuron whose job is to stop the model from writing a period where one would be
wrong. We can find such neurons **from the weights alone**. A neuron's output vector goes, through the final
normalization's weights γ, straight to the vocabulary: its **direct logit effect** is W_U(γ ⊙ w_down), one number per
token. Its 10 most negative tokens are what it pushes *against*.

The lecture counted, over all 172,032 neurons, how many of the 10 most-suppressed tokens begin with ".". 79 neurons have
at least 8 of 10; 45 have all 10. Two of them, layer 26 neurons 3926 and 1424, are clean period suppressors.

In [ ]:
V = len(tok)
WU = lm.lm_head.weight[:V]                                   # one row per vocabulary entry
gam = lm.model.norm.weight
strs = [tok.decode([i]) for i in range(V)]
is_dot = torch.tensor([s.startswith(".") for s in strs], device=DEV)
print(f"{int(is_dot.sum()):,} of {V:,} vocabulary entries begin with '.'")
ndot = torch.zeros(len(LAYERS), 6144, dtype=torch.long)
dot_eff = torch.zeros(len(LAYERS), 6144)
t0 = time.time()
with torch.no_grad():
    for li, layer in enumerate(LAYERS):
        D = layer.mlp.down_proj.weight                       # 2048 x 6144: each column is one neuron's output
        for c in range(0, 6144, 1536):
            E = WU @ (gam[:, None] * D[:, c:c + 1536])       # vocabulary x 1536 neurons
            E = E - E.mean(0, keepdim=True)                  # softmax ignores a shift shared by every token
            dot_eff[li, c:c + 1536] = E[is_dot].mean(0).cpu()
            bot = E.topk(10, dim=0, largest=False).indices    # 10 most suppressed tokens per neuron
            ndot[li, c:c + 1536] = is_dot[bot].sum(0).cpu()
print(f"every neuron's logit effect in {time.time() - t0:.0f} s")
check("vocabulary entries that begin with '.'", int(is_dot.sum()), 6100)
check("neurons with at least 8 of 10 dot-initial", int((ndot >= 8).sum()), 79, tol=2)
check("neurons with all 10 dot-initial", int((ndot == 10).sum()), 45, tol=2)
check("L26/N3926: dot-initial among its 10 most suppressed", int(ndot[26, 3926]), 10)
check("L26/N3926: its mean effect on dot tokens", float(dot_eff[26, 3926]), -0.2687, tol=0.003)
with torch.no_grad():
    E = WU @ (gam * LAYERS[26].mlp.down_proj.weight[:, 3926])
print("L26/N3926 pushes hardest against:", [strs[i] for i in E.topk(10, largest=False).indices.tolist()])

Now watch it work. The lecture found neuron 3926 fires on the middle dot of a *spaced* ellipsis (". . ."), where the next
token is " ." with a space, so a bare "." would be wrong. Removing the neuron there made a bare period about 11 times
likelier. Here are some spaced ellipses of our own.

In [ ]:
PROMPTS = ["I am not sure . . . maybe tomorrow.", "Well . . . that was unexpected.", "He waited . . . and waited.",
           "The answer is . . . nobody knows.", "And then . . . silence.", "We tried, and tried . . . nothing worked.",
           "She opened the door . . . and froze.", "It was late . . . very late.", "Count them: one . . . two . . . three.",
           "So the story goes . . . or so they say."]
DOTS = is_dot


def p_dot_after_middle(text, ablate):
    ids_ = tok(text, return_tensors="pt")["input_ids"].to(DEV)
    t = [tok.decode([i]) for i in ids_[0].tolist()]
    spots = [k for k in range(1, len(t) - 1) if t[k].strip() == "." and t[k - 1].strip() == "." and t[k + 1].strip() == "."]
    fire = {}
    def hook(m, inp):
        fire["a"] = inp[0][0, :, 3926].clone()
        if ablate:
            x = inp[0].clone(); x[..., 3926] = 0
            return (x,)
    h = LAYERS[26].mlp.down_proj.register_forward_pre_hook(hook)
    with torch.no_grad():
        p = torch.softmax(lm(ids_).logits[0, :, :V].float(), -1)   # the output layer has a few unused rows past the vocabulary
    h.remove()
    return [(float(p[k][DOTS].sum()), float(fire["a"][k])) for k in spots]


base = [r for t in PROMPTS for r in p_dot_after_middle(t, False)]
abl = [r for t in PROMPTS for r in p_dot_after_middle(t, True)]
fires = np.mean([a for _, a in base]) / 480.0
ratio = np.mean([p for p, _ in abl]) / np.mean([p for p, _ in base])
print(f"{len(base)} middle dots; 3926 fires at {fires:.0%} of its strongest on average")
print(f"P(next token begins with '.'): {np.mean([p for p, _ in base]):.4%} -> {np.mean([p for p, _ in abl]):.4%} without the neuron (x{ratio:.1f})")
check("removing 3926 makes a bare '.' likelier there (more than twice)", bool(ratio > 2), True)

### Your turn 3 · The other suppressor

Layer 26 neuron 1424 is the lecture's second clean suppressor (its strongest suppressed tokens include ".).", ".", ".\")."
and ".';"). Check its count of dot-initial tokens among its 10 most suppressed, and its mean effect on dot tokens.

*Hint: `ndot[26, 1424]`, `dot_eff[26, 1424]`.*

In [ ]:
# TODO: neuron 1424's dot count and dot effect

## Plate XII · Weak, in words

The essay says almost all the weak activations of a network can be removed for almost nothing. The lecture tested it one
layer at a time. Here we take the cut it calls **p99**: in one layer, keep only each neuron's strongest 1% of values in
each direction (above its 99th percentile or below its 1st) and zero the other 98%. The percentiles come from 16
calibration sequences; the loss (the average surprise at the true next token, in nats) from 16 others.

The lecture, on 128 sequences: layer 0 costs +0.067 nats, layer 17 only +0.017, and the last layer, 27, the most:
+0.140. Removal is cheap in the middle and costs most at the two ends.

In [ ]:
def acts(layer_ids, data):
    """Every value a of the given layers on these sequences (position 0 skipped), as float16 on the CPU."""
    store = {l: [] for l in layer_ids}
    hk = [LAYERS[l].mlp.down_proj.register_forward_pre_hook(lambda m, inp, l=l: store[l].append(inp[0][:, 1:].half().cpu()))
          for l in layer_ids]
    with torch.no_grad():
        for s in range(0, len(data), 4):
            lm(data[s:s + 4].to(DEV))
    for h in hk:
        h.remove()
    return {l: torch.cat(v).reshape(-1, 6144) for l, v in store.items()}


def loss(data, layer=None, keep_fn=None):
    hk = []
    if layer is not None:
        def cut(m, inp):
            a = inp[0]
            return (torch.where(keep_fn(a), a, torch.zeros_like(a)),)
        hk.append(LAYERS[layer].mlp.down_proj.register_forward_pre_hook(cut))
    tot, n = 0.0, 0
    with torch.no_grad():
        for s in range(0, len(data), 4):
            x = data[s:s + 4].to(DEV)
            lp = torch.log_softmax(lm(x).logits[:, :-1].float(), -1)
            tot += float(-lp.gather(-1, x[:, 1:, None]).sum()); n += x[:, 1:].numel()
    for h in hk:
        h.remove()
    return tot / n


CUT_LAYERS = [0, 17, 27]
cal = acts(CUT_LAYERS, CALIB)
q = {l: (torch.quantile(cal[l][::4].float(), 0.99, dim=0).to(DEV), torch.quantile(cal[l][::4].float(), 0.01, dim=0).to(DEV))
     for l in CUT_LAYERS}
base_loss = loss(HELD)
print(f"held-out loss {base_loss:.3f} nats ({HELD[:, 1:].numel():,} tokens)")
delta = {}
for l in CUT_LAYERS:
    hi, lo = q[l]
    delta[l] = loss(HELD, l, lambda a, hi=hi, lo=lo: (a >= hi) | (a <= lo)) - base_loss
    print(f"layer {l:2d}: keep only the top and bottom 1% -> loss {minus(f'{delta[l]:+.3f}')} nats")
check("layer 0, cost of the p99 cut", delta[0], 0.0667, tol=0.04)
check("layer 17, cost of the p99 cut", delta[17], 0.0174, tol=0.03)
check("layer 27, cost of the p99 cut", delta[27], 0.1395, tol=0.05)
check("the last layer costs more than the middle one", delta[27] > delta[17], True)

Each layer alone is cheap: zeroing 98% of a layer's values costs a few hundredths of a nat to a tenth, most at the two
ends of the network. But the lecture also cut *every* layer at once, and the model fell apart (+6.9 nats for the
"below 10% of the extreme" cut). The essay's claim holds layer by layer, not for the whole network.

### Your turn 4 · The other cut

The lecture's third cut zeroes only the −×− regime (gate closed, value negative) in one layer. For layer 0 it cost
+0.022 nats. Measure it.

*Hint: you need the gate's sign inside the hook. Hook `gate_proj` to remember it, then zero `a` where both are negative.*

In [ ]:
# TODO: zero the -x- regime in layer 0 and measure the held-out loss

## Part C · The ledger (Plates XIV–XVI)

The lecture's verdict rests on 23 results from across the course, each copied from its lecture's findings and pinned to
one of the essay's claims, with which way it leans. Here they are as data.

In [ ]:
LEDGER = [["one", "L10", "for"], ["one", "L17", "for"], ["one", "L12", "for"], ["one", "L30a", "for"], ["one", "L30b", "for"], ["one", "L15", "against"], ["one", "L16", "against"], ["one", "L29", "against"], ["tools", "L12", "for"], ["tools", "L28", "for"], ["tools", "L30b", "mixed"], ["tools", "L27", "against"], ["tools", "L30a", "mixed"], ["compounds", "L10", "for"], ["compounds", "L19", "against"], ["gaps", "L17", "for"], ["gaps", "L24", "for"], ["gaps", "L22", "for"], ["basis", "L26", "for"], ["basis", "L26", "mixed"], ["basis", "L14", "against"], ["basis", "L14", "against"], ["one", "L12", "for"]]     # (claim, lecture, lean)
CLAIMS = {"one": "one learning principle", "tools": "modern tools work", "compounds": "understanding compounds",
          "gaps": "bad tools, explanations, learning", "basis": "neurons are the right units"}
from collections import Counter
tally = Counter((c, lean) for c, _, lean in LEDGER)
for c, name in CLAIMS.items():
    print(f"{name:36s}", "  ".join(f"{lean} {tally[(c, lean)]}" for lean in ("for", "mixed", "against") if tally[(c, lean)]))
check("results in the ledger", len(LEDGER), 23)
check("results that lean the essay's way", sum(lean == "for" for _, _, lean in LEDGER), 13)

### Your turn 5 · Where the essay is weakest

Which claim has the most results against it? And which lectures do those results come from?

*Hint: filter `LEDGER` by lean.*

In [ ]:
# TODO: the claim with the largest share of 'against' rows, and their lectures

### The verdict

On real networks the evidence leans the essay's way: one learning principle, modern tools that mostly work, and
understanding that compounds lecture by lecture. Packing (superposition) is real, but we found it only in toys built too
small, like Part A's. The neuron as the unit is mixed: a faithful sparse autoencoder copies neurons back, yet random
directions read about as well as neurons in a blind test. And the one neuron the essay could not finish, 5a:719, is
Lecture 31's.

## Solutions

**Your turn 1 · When does the fifth feature join?**

In [ ]:
S_try = (0.80, 0.82, 0.84, 0.86, 0.88)
more = train_toy(S_list=S_try, seeds=10)
k_try = {S: int((np.linalg.norm(Wb[0], axis=0) > 0.5).sum()) for S, Wb in more.items()}
print(k_try)
first5 = min(S for S, k in k_try.items() if k == 5)
check("first sparsity tried that keeps all five", first5 in (0.82, 0.84, 0.86), True)

Between 0.80 and about 0.84, as in Lecture 15; seeds disagree near the transition, which is why the check allows a step either side.

**Your turn 2 · Is the neuron typical?**

In [ ]:
open_share = (counts[..., 2] + counts[..., 3]) / counts.sum(-1)
per_layer = (open_share < 0.10).float().mean(1).cpu().numpy()
for l in range(len(per_layer)):
    print(f"layer {l:2d}: {per_layer[l]:.0%} of neurons open their gate on fewer than 10% of tokens")
check("median neuron's gate-open share (lecture: 10.7%)", float(open_share.median()), 0.107, tol=0.03)

Most neurons keep their gate shut on most tokens. Layers 1–4 have the most nearly-silent gates (over 80% of their neurons open on fewer than one token in ten), and so does the last layer; layer 0 and the middle layers (12–16) open far more often. That is why the noisy middle is −×− almost everywhere.

**Your turn 3 · The other suppressor.**

In [ ]:
check("L26/N1424: dot-initial among its 10 most suppressed", int(ndot[26, 1424]), 10)
check("L26/N1424: its mean effect on dot tokens", float(dot_eff[26, 1424]), -0.3456, tol=0.003)

**Your turn 4 · The other cut.**

In [ ]:
sign = {}
h1 = LAYERS[0].mlp.gate_proj.register_forward_hook(lambda m, i, o: sign.__setitem__("g", o))
def no_negneg(a):
    return ~((sign["g"] < 0) & (a > 0))         # -x-: gate closed and a > 0 (so the value was negative)
d0 = loss(HELD, 0, no_negneg) - base_loss
h1.remove()
print(f"layer 0 without its -x- values: {minus(f'{d0:+.3f}')} nats")
check("layer 0, cost of zeroing -x-", d0, 0.0221, tol=0.03)

Cheap: the double-negative middle carries little. Layer 0's +0.02 is the largest of any layer in the lecture.

**Your turn 5 · Where the essay is weakest.**

In [ ]:
for c, name in CLAIMS.items():
    rows_c = [r for r in LEDGER if r[0] == c]
    against = [lec for _, lec, lean in rows_c if lean == "against"]
    print(f"{name:36s} {len(against)} of {len(rows_c)} against: {', '.join(against)}")
worst = max(CLAIMS, key=lambda c: sum(r[0] == c and r[2] == "against" for r in LEDGER))
check("the claim with the most results against it", worst, "one")
check("where they come from", sorted(lec for c, lec, lean in LEDGER if c == "one" and lean == "against"), ["L15", "L16", "L29"])

One learning principle has the most rows against it, three, and all three are toy networks built with too few neurons (L15, L16, L29), where packing really happens. On real networks the claim leans the essay's way. The shakiest claim by share is the neuron as the unit: two of its four rows lean against it, both from L14's blind tests.